# MiniGPT from scratch — Colab demo

Character-level GPT (0.81M parameters) trained from scratch on Tiny Shakespeare. No pretrained weights, no `transformers` model classes.

**Runtime → Change runtime type → T4 GPU** is optional: everything below works on CPU; only re-training (section 5) benefits from a GPU.

Sections: 1 setup · 2 tests · 3 architecture checks · 4 inference from the committed checkpoint · 5 (optional) re-train · 6 evaluation & attention · 7 API

In [ ]:
# 1. Setup: clone the repository and install dependencies
REPO_URL = "https://github.com/laibaa1209/AI-ENG.git"
!git clone -q $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2. Unit tests (attention, causal mask, shapes, tiny overfit gate, API)

In [ ]:
!python -m pytest -q

## 3. Architecture checks: hand-calculated attention, multi-head shape trace, positional encoding

In [ ]:
!python -m experiments.hand_attention
!python -m experiments.shape_trace
!python -m experiments.positional_encoding_viz
from IPython.display import Image, display
for f in ["fig2_causal_mask_hand", "fig4_positional_encoding"]:
    display(Image(f"outputs/figures/{f}.png"))

## 4. Inference from the committed checkpoint (`checkpoints/tiny_gpt.pt`) — three fixed prompts

In [ ]:
!python -m src.generate --prompt "ROMEO:" --max-new-tokens 200
!python -m src.generate --prompt "JULIET:" --max-new-tokens 200
!python -m src.generate --prompt "First Citizen:" --max-new-tokens 200

## 5. (Optional) Re-train from scratch

Same configuration as the committed checkpoint. ~5 min on a T4 GPU, ~70 min on a laptop CPU.
Written to a separate checkpoint so the committed one is untouched.

In [ ]:
!python -m src.train --config configs/tiny_gpt.yaml --set run_name=colab_gpt checkpoint_path=checkpoints/colab_gpt.pt
display(Image("outputs/figures/loss_curve_colab_gpt.png"))
!python -m src.generate --checkpoint checkpoints/colab_gpt.pt --prompt "ROMEO:"

## 6. Evaluation (full validation perplexity, 5 fixed prompts, greedy vs temperature) and attention maps of the trained model

In [ ]:
!python -m experiments.evaluate
!python -m experiments.attention_visualization
display(Image("outputs/figures/loss_curve_tiny_gpt.png"))
display(Image("outputs/figures/fig7_attention_trained.png"))
display(Image("outputs/figures/ablation_val_curves.png"))
print(open("outputs/logs/ablation_results.md").read())

## 7. REST API (FastAPI) — start the server in the background and call it

In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "api.app:app", "--port", "8000"])
time.sleep(8)
print(requests.get("http://127.0.0.1:8000/health").json())
r = requests.post("http://127.0.0.1:8000/generate", json={"prompt": "ROMEO:", "max_new_tokens": 150, "seed": 0})
print(r.json()["generated"])
server.terminate()